# Objective
The objective of this notebook is to take the very detailed ground truth results for the problem instances, which contain the exact scores of the ensembles for different values for $b$ and $t$, and aggregate them into the estimates of the ground truth of $\mathbb{E}[Z_{nt}]$, $\mathbb{V}[Z_{nt}]$, $\mathbb{E}[Z_{nt}|D_{app}]$, and $\mathbb{V}[Z_{nt}|D_{app}]$.

In [25]:
import json
import gzip
import os
import pathlib
from experiments.problem_instance.problem_instance import ProblemInstance
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from tqdm import tqdm

In [26]:
FOLDER = "/home/felix/pCloudDrive/research/asforests/problem instances with detailed scores"

In [27]:
rows = []
for name in os.listdir(FOLDER):    
    parts = [int(p) for p in name[:-8].split("_")]
    rows.append(parts)
df_overview = pd.DataFrame(rows, columns=["openmlid", "data_seed", "num_trees", "val_size"])
df_overview

,openmlid,data_seed,num_trees,val_size
0,1111,3,1,2
1,1111,3,256,2
2,1111,3,64,2
3,1111,3,8,2
4,1111,4,1,256
...,...,...,...,...
2255,1461,9,256,64
2256,1461,9,64,256
2257,1461,9,64,64
2258,1461,9,8,256


In [28]:
def plot_approximation(d):
    scores_iid = np.array(d["scores_iid"])

    fig, axs = plt.subplots(1, scores_iid.shape[1] + 1, figsize=(20, 4), sharey=True)

    for idx_n, n in enumerate(d["n_checkpoints"]):
        ax = axs[idx_n]
        out = []
        for idx_t, t in enumerate(d["t_checkpoints"]):
            out.append(scores_iid[:, idx_n, idx_t])
        ax.boxplot(out)
        ax.set_yscale("log")
        ax.set_title(f"{n=}")
        ax.grid()

    scores_cond = np.array(d["scores_cond"])

    ax = axs[-1]
    out = []
    for idx_t, t in enumerate(d["t_checkpoints"]):
        out.append(scores_cond[:, idx_t])
    ax.boxplot(out)
    ax.grid()
    fig.suptitle(d["data_description"])
    plt.show()

In [37]:
target_folder_instances = "instances"
target_folder_instance_groups = "merged instances"

for folder in [target_folder_instances, target_folder_instance_groups]:
    pathlib.Path(folder).mkdir(parents=True, exist_ok=True)

for (openmlid, num_trees, val_instances), df_case in tqdm(df_overview.groupby(["openmlid", "num_trees", "val_size"])):

    print(openmlid, num_trees, val_instances)
    instances = []
    any_skipped = False
    for seed, df_seed in df_case.groupby("data_seed"):
        print("\t", seed)
        name = f"{openmlid}_{seed}_{num_trees}_{val_instances}"
        target_file = f"{target_folder_instances}/{name}.json"

        if not pathlib.Path(target_file).exists():

            with gzip.open(f"{FOLDER}/{name}.json.gz", "rt", encoding="utf-8") as f:
                d = json.load(f)
                scores_iid = np.array(d.pop("scores_iid"))
                scores_cond = np.array(d.pop("scores_cond"))
                val_instances_per_class = d.pop("validation_instances_per_class")
                assert val_instances_per_class == val_instances
                d["true_means_for_iid_case"] = scores_iid.mean(axis=(0, 1))
                d["true_vars_for_iid_case"] = scores_iid.var(axis=0)
                d["true_means_for_cond_case"] = scores_cond.mean(axis=0)
                d["true_vars_for_cond_case"] = scores_cond.var(axis=0)
                pi = ProblemInstance.from_dict(d)
                d = pi.to_dict()

                with open(target_file, "w") as f:
                    json.dump(d, f)
                    instances.append(d)

        else:
            any_skipped = True
    
    if not any_skipped:
        with open(f"{target_folder_instance_groups}/{openmlid}_{num_trees}_{val_instances}.json", "w") as f:
            json.dump(instances, f)

  0%|          | 0/235 [00:00<?, ?it/s]

 14%|█▎        | 32/235 [00:00<00:00, 319.32it/s]

3 1 2
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
3 1 8
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
3 1 64
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
3 1 256
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
3 8 2
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
3 8 8
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
3 8 64
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
3 8 256
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
3 64 2
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
3 64 8
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
3 64 64
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
3 64 256
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
3 256 2
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
3 256 8
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
3 256 64
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
3 256 256
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
12 1 2
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
12 1 8
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
12 1 64
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
12 1 256
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
12 8 2
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
12

 40%|████      | 94/235 [00:00<00:00, 288.52it/s]

	 3
	 4
	 5
	 6
	 7
	 8
	 9
31 256 256
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
54 1 2
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
54 1 8
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
54 1 64
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
54 1 256
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
54 8 2
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
54 8 8
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
54 8 64
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
54 8 256
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
54 64 2
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
54 64 8
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
54 64 64
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
54 64 256
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
54 256 2
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
54 256 8
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
54 256 64
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
54 256 256
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
181 1 2
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
181 1 8
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
181 1 64
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9


...

1457 256 64
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9


 74%|███████▍  | 175/235 [04:44<13:53, 13.89s/it]

1457 256 256
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9


100%|██████████| 235/235 [05:09<00:00,  1.32s/it]

1461 1 2
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
1461 1 8
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
1461 1 64
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
1461 1 256
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
1461 8 2
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
1461 8 8
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
1461 8 64
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
1461 8 256
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
1461 64 2
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
1461 64 8
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
1461 64 64
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
1461 64 256
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
1461 256 2
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
1461 256 8
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
1461 256 64
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
1461 256 256
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
1464 1 2
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
1464 1 8
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
1464 1 64
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	 8
	 9
1464 1 256
	 0
	 1
	 2
	 3
	 4
	 5
	 6
	 7
	